In [4]:
from openpyxl import load_workbook
import sys

# 日付が列3～列Xまで存在する
# 戻り値は最後の列番号を数値にしたもの(last_col)
def last_col(ws):
    col = 3   # C列から始める
    while ws.cell(row=2, column=col).value is not None:
        col += 1
    last_col = col - 1   # 空欄の1つ手前が最後の列
    return last_col


# 前月末在庫を決める関数
# 戻り値はpremonth_stock_list, pre_flag
def get_premonth_stock(wb, y, m, target_ws):
    # 対象のワークシートの一か月前のワークシートをpre_target_wsへ代入
    # 対象月が1月の場合は前月は12月で、年はy-1
    if m == 1:
        pre_target_ws_name = f"{y-1}.{12}"
    else:
        # 対象月が1月以外の場合は前月はm-1、年はy
        pre_target_ws_name = f"{y}.{m-1}"
    # 前月のワークシートの名前がシートネームの中になかったら
    if pre_target_ws_name not in wb.sheetnames:
        # 前月フラグはFalse
        pre_flag = False
        # 前月末在庫のリストを初期化して作成
        premonth_stock_list = [0, 0, 0]
        # idxを0-2までループしてインクリメント
        # 0:部品A　1:部品B　2:部品C
        for idx in range(3):
            # premonth_stock_listへ部品A,B,Cの前月末在庫の数値(処理対象シートのB12-B14)を代入する
            premonth_stock_list[idx] = target_ws[f"B{12 + idx}"].value
            # 前月末在庫の値がint型ではなかった場合、エラーメッセージを表示してプログラム終了
            if not isinstance(premonth_stock_list[idx], int):
                print("前月末在庫（B12〜B14）を入力してください")
                sys.exit(1)
    # 前月のワークシートの名前がシートネームの中にあったら
    else:
        # 前月フラグはTrue
        pre_flag = True
        # "前月_出力"シートがなかった場合、エラーメッセージを表示してプログラム終了
        pre_target_ws_name = pre_target_ws_name + "_出力"
        if pre_target_ws_name not in wb.sheetnames:
            print(f"{pre_target_ws_name}がありません。前月を先に実行してください。")
            sys.exit(1)
        # "前月_出力"シートがあった場合
        else:
            pre_target_ws = wb[pre_target_ws_name]
            # 前月の日付の最終列を求めてpre_last_colへ最終列を数値として代入
            pre_last_col = last_col(pre_target_ws)
            # 前月末在庫のリストを初期化して作成
            premonth_stock_list = [0, 0, 0]
            # idxを0-2までループしてインクリメント
            # 0:部品A　1:部品B　2:部品C
            for idx in range(3):
                # premonth_stock_listへ部品A,B,Cの前月末在庫の数値(前月シートの最終日の部品A,B,Cの各在庫量)を代入する
                premonth_stock_list[idx] = pre_target_ws.cell(row=12 + idx, column=pre_last_col).value
                # 前月末在庫の値がint型ではなかった場合、エラーメッセージを表示してプログラム終了
                if not isinstance(premonth_stock_list[idx], int):
                    print("前月シートの前月末在庫の値が不正です。")
                    sys.exit(1)
    return premonth_stock_list, pre_flag


# 部品クラスの宣言
class Part:
    def __init__(self, idx, premonth_stock, target_ws):
        self.idx = idx
        self.target_ws = target_ws
        self.order_cnt = 0
        self.threshold, self.order_qty = self.input_default()
        self.premonth_stock = premonth_stock
        self.stock_dict = {}
        self.prod_dict = {}
        self.last_column = last_col(target_ws)

    # 閾値と発注量を変数に代入
    # 戻り値：閾値(threshold)と発注量(order_qty)
    def input_default(self):
        if self.idx not in (0, 1, 2):
            print("Partクラスの引数の指定が異常です。")
            sys.exit(1)
        # 処理対象シートのB17-B19, C17-C19を参照して引数idxに応じて適切な値を代入する
        threshold = self.target_ws[f"B{17 + self.idx}"].value
        order_qty = self.target_ws[f"C{17 + self.idx}"].value
        return threshold, order_qty


    # 生産量と在庫量を計算してprod_dictとstock_dictに代入する。
    # 戻り値なし
    def count(self):
        # 前日在庫(preday_stock)に前月末在庫(self.premonth_stock)を代入する
        preday_stock = self.premonth_stock
        # col=3(C列)～最終日の列までループしてcolをインクリメント
        for col in range(3, self.last_column + 1):
            # 消費量(cons)に処理対象シートの各部品の消費量を代入する(idxで部品A,B,Cを決める)
            cons = self.target_ws.cell(row=4 + self.idx, column=col).value
            # 在庫量 = 前日在庫 - 消費量
            stock = preday_stock - cons
            # 日付が土日だったら生産量は0
            if self.target_ws.cell(row=2, column=col).value.weekday() > 4:
                prod = 0
            # 日付が平日だったら
            else:
                # 前日在庫 <= 閾値だったら指定の発注量を発注する
                if preday_stock <= self.threshold:
                    # 生産量 = 発注量
                    prod = self.order_qty
                    # 在庫数 = 在庫数 + 発注量
                    stock = stock + prod
                    # 発注回数をインクリメントする
                    self.order_cnt += 1
                # 前日在庫 > 閾値だったら
                else:
                    # 生産量 = 0
                    prod = 0
            # 生産量ディクショナリ(prod_dict[])に生産量を代入する→処理としては1日から最終日まで繰り返す
            self.prod_dict[col]= prod
            # 在庫量ディクショナリ(stock_dict[])に在庫量を代入する→処理としては1日から最終日まで繰り返す
            self.stock_dict[col] = stock
            # 前日在庫 = 在庫
            preday_stock = stock

    # 出力のエクセルファイルに情報を書き込む関数
    def write_ws(self, ws_out, pre_flag):
        # C列～最終日の列までループでインクリメントし、各日の生産量と在庫量をすべて書き込む
        for col in range(3, self.last_column + 1):
            ws_out.cell(row=8 + self.idx, column=col).value = self.prod_dict[col]
            ws_out.cell(row=12 + self.idx, column=col).value = self.stock_dict[col]
        # 前月シートがあった場合
        if pre_flag == True:
            # 前月末在庫をB12-B14へ書き込む
            ws_out.cell(row=12 + self.idx, column = 2).value = self.premonth_stock


# 作成した出力シートを適切な位置へ配置する関数(処理対象シートのすぐ後ろに出力シートを配置する)
# 戻り値なし
def set_pos_ws_out(wb, target_ws, output_ws_name, ws_out):
    # 処理対象シートの位置を取得
    target_pos = wb.sheetnames.index(target_ws.title)
    # 作成した出力シートの位置を取得
    output_pos = wb.sheetnames.index(output_ws_name)
    # 両シートの差をoffsetに代入
    offset = output_pos - target_pos
    # 左へoffset-1だけ移動
    move = -(offset - 1)
    wb.move_sheet(ws_out, move)


# メイン関数
def main():
    wb = load_workbook("./生産計画自動計算.xlsx")
    ws = wb["処理対象"]   # シートを取り出す
    y = ws["A1"].value    # 対象の年をyへ代入
    m = ws["A2"].value    # 対象の月をmへ代入
    target_ws = wb[f"{y}.{m}"]   # 処理対象シートをtarget_wsへ代入
    output_ws_name = f"{y}.{m}" + "_出力"  # 出力するシート名

    # 前月末在庫リスト、前月シート有無フラグ(pre_flag)へ値を代入
    premonth_stock_list, pre_flag = get_premonth_stock(wb, y, m, target_ws)

    ##### 出力シートの作成と配置 #####
    # 出力シートがワークシートネームにあった場合
    if output_ws_name in wb.sheetnames:
        # 出力シートを削除
        wb.remove(wb[output_ws_name])
    # 処理対象シートをコピーしてws_outへ代入
    ws_out = wb.copy_worksheet(target_ws)
    # ws_outのシート名を"処理対象シート_出力"に書き換える
    ws_out.title = output_ws_name
    # 作成した出力シートを適切な位置へ配置する
    set_pos_ws_out(wb, target_ws, output_ws_name, ws_out)

    ##### 部品のインスタンス化、生産量・在庫量・前月末在庫の計算、出力ファイルへの書き込み、発注回数の表示 #####
    order_cnt = {} # 発注回数のディレクトリ
    # (idx: 部品) = (0: A), (1: B), (2: C)の関係
    for idx in range(3):
        # 部品A, B, Cをインスタンス化する
        p = Part(idx, premonth_stock_list[idx], target_ws)
        # 生産量と在庫量を計算してprod_dictディクショナリとstock_dictディクショナリに代入する(一ヶ月分)。
        p.count()
        # 出力のエクセルファイルに生産量・在庫量・前月末在庫などすべての必要な情報を書き込む
        # pre_flagがFlaseの時はシートをコピーした時点で前月末在庫に値が入っているため、書き込む必要はない
        p.write_ws(ws_out, pre_flag)
        # 発注回数をorder_cntディレクトリに代入
        order_cnt[idx] = p.order_cnt
    # エラー処理
    try:
        wb.save("生産計画自動計算.xlsx")
        print(f"{output_ws_name}を作成しました。")
        part_name = ["A", "B", "C"]
        for idx, kind in enumerate(part_name):
            print(f"部品{kind}の発注：{order_cnt[idx]}回")
    except PermissionError:
        print("保存できません。Excelでファイルを閉じてから実行してください。")
        sys.exit(1)

# このファイルを直接実行したときだけ main() を呼び出す、というお決まりの書き方
if __name__ == "__main__":
    main()


2026.10_出力を作成しました。
部品Aの発注：7回
部品Bの発注：8回
部品Cの発注：12回
